In [ ]:
from project_config import input_path, output_path, table_path, require_columns, BOOTSTRAP_RESAMPLES, RANDOM_SEED


In [ ]:
#Author Alex J. Yang, alexjieyang@outlook.com
import numpy as np
import pandas as pd
import itertools
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm

In [ ]:
dfp = pd.read_csv(input_path('Paper.csv.zip'))

In [ ]:
required_columns = ['PaperID', 'Year', 'C5', 'Citation_Count', 'hit', 'Reference_Count',
       'Team_Size', 'Institution_Count', 'Country_Count', 'Title_Length',
       'Atyp_10pct_Z', 'Novelty_percentile', 'Disruption',
       'Disruption_percentile', 'RaoStirling', 'NIH_Count', 'NSF_Count',
       'Patent_Count', 'Newsfeed_Count', 'Tweet_Count', 'NCT_Count',
       'TeamAgeDiversity', 'Team_Diversity_Gini', 'TeamAgeMean', 'TeamAgeMax',
       'TeamAgeMin', 'Reference_Age_Mean', 'Reference_Age_Std', 'FieldID',
       'FieldName', 'FieldID2', 'FieldName2', 'JournalID']
require_columns(dfp, required_columns)


In [ ]:
Flst = ['Mathematics', 'Physics','Materials science', 'Engineering', 'Computer science',
       'Geography', 'Geology', 'Environmental science', 'Chemistry',
        'Medicine', 'Biology','Psychology', 'Economics', 'Business', 'Political science','Sociology']
dfp = dfp[dfp['FieldName'].isin(Flst)]

In [ ]:
dfp = dfp[dfp['TeamAgeMean']<60].copy()
dfp.index = range(len(dfp))

# Draw Fig. 1 a-c for 16 fields

In [ ]:
colors = sns.color_palette("mako", 16)

In [ ]:
x = range(1950 , 2017)
c = 'TeamAgeMean'
y = dfp.groupby('Year')[c].mean()
ylst = [dfp[dfp['FieldName']==F].groupby('Year')[c].mean() for F in Flst]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
for i in range(len(ylst)):
    plt.plot(ylst[i].index, ylst[i].values,linewidth=1,alpha = .4,c=colors[i])
    sns.lineplot(dfp[dfp['FieldName']==Flst[i]],x='Year',y=c,color=colors[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)

plt.plot(y.index, y.values,linewidth=3,alpha = 1,c='black')
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Age',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(1,15)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/a.svg'),bbox_inches = 'tight')

In [ ]:
x = range(1950 , 2017)
c2 = 'TeamAgeDiversity'
y2 = dfp.groupby('Year')[c2].mean()
ylst2 = [dfp[dfp['FieldName']==F].groupby('Year')[c2].mean() for F in Flst]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))

for i in range(len(ylst2)):
    plt.plot(ylst2[i].index, ylst2[i].values,linewidth=1,alpha = .4,c=colors[i])
    sns.lineplot(dfp[dfp['FieldName']==Flst[i]],x='Year',y=c2,color=colors[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.plot(y2.index, y2.values,linewidth=3,alpha = 1,c='black')
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Age Diverisity',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(0.48,1.02)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/b.svg'),bbox_inches = 'tight')

In [ ]:
d1 = pd.DataFrame(dfp.groupby(dfp['FieldName'])[c].mean())  
d2 = pd.DataFrame(dfp.groupby(dfp['FieldName'])[c2].mean())
d1 = d1.loc[Flst,:]
d2 = d2.loc[Flst,:]

In [ ]:
plt.subplots(1,2,figsize=(5,5))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.2,hspace=0)
ax = plt.subplot(1,2, 1)
plt.barh( d1.index ,  width=d1.values.flatten(), capsize = 3, color = colors)
plt.xlim(6,12)
plt.ylim(-.7,15.7)
plt.yticks(fontsize=20,)
plt.xticks([6,8,10],fontsize=20,)
plt.title('Team Age  ',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

ax = plt.subplot(1,2, 2)
plt.barh( d2.index ,  width=d2.values.flatten(), capsize = 3, color = colors)
plt.xlim(0.6,1)
plt.ylim(-.7,15.7)
plt.yticks(fontsize=0,)
plt.xticks([.6,.8,1],fontsize=20,)
plt.title('  Team Age Diverisity',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/c.svg'),bbox_inches = 'tight')

# # Draw Fig. 1 d-e for different team sizes

In [ ]:
def resetTeam(x):
    if x<9:
        return x
    else:
        return 9
dfp['TeamSizeG'] = dfp['Team_Size'].apply(resetTeam)

In [ ]:
colors2 = sns.color_palette("rocket", 9)

In [ ]:
x = range(1950 , 2017)
c = 'TeamAgeMean'
ylst3 = [dfp[dfp['TeamSizeG']==G].groupby('Year')[c].mean() for G in range(1,10)]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
plt.plot(y.index, y.values,linewidth=3,alpha = 1,c='black')
for i in range(len(ylst3)):
    plt.plot(ylst3[i].index, ylst3[i].values,linewidth=2,alpha = 1,c=colors2[i])
    sns.lineplot(dfp[dfp['TeamSizeG']==i+1],x='Year',y=c,color=colors2[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Age',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(1,15)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/e.svg'),bbox_inches = 'tight')

In [ ]:
x = range(1950 , 2017)
c2 = 'TeamAgeDiversity'
ylst4 = [dfp[dfp['TeamSizeG']==G].groupby('Year')[c2].mean() for G in range(1,10)]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
plt.plot(y2.index, y2.values,linewidth=3,alpha = 1,c='black')
for i in range(len(ylst4)):
    plt.plot(ylst4[i].index, ylst4[i].values,linewidth=2,alpha = 1,c=colors2[i])
    sns.lineplot(dfp[dfp['TeamSizeG']==i+1],x='Year',y=c2,color=colors2[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Age Diverisity',fontsize = 20)
plt.ylim(0.48,1.02)
plt.xlim(1950,2016)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/f.svg'),bbox_inches = 'tight')

In [ ]:
d11 = pd.DataFrame(dfp.groupby(dfp['TeamSizeG'])[c].mean())  
d21 = pd.DataFrame(dfp.groupby(dfp['TeamSizeG'])[c2].mean())

In [ ]:
d11.index = ['Team Size = 1','Team Size = 2','Team Size = 3','Team Size = 4','Team Size = 5','Team Size = 6','Team Size = 7','Team Size = 8','Team Size > 8']
d21.index = ['Team Size = 1','Team Size = 2','Team Size = 3','Team Size = 4','Team Size = 5','Team Size = 6','Team Size = 7','Team Size = 8','Team Size > 8']

In [ ]:
plt.subplots(1,2,figsize=(5,5))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.2,hspace=0)
ax = plt.subplot(1,2, 1)
plt.barh( d11.index ,  width=d11.values.flatten(), capsize = 3, color = colors2)
plt.xlim(6,14)
plt.ylim(-.7,8.7)
plt.yticks(fontsize=20,)
plt.xticks([6,8,10,12],fontsize=20,)
plt.title('Team Age  ',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

ax = plt.subplot(1,2, 2)
plt.barh( d21.index ,  width=d21.values.flatten(), capsize = 3, color = colors2)
plt.xlim(0.6,1)
plt.ylim(-.7,8.7)
plt.yticks(fontsize=0,)
plt.xticks([.4,.6,.8,1],fontsize=20,)
plt.title('  Team Age Diverisity',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/g.svg'),bbox_inches = 'tight')

# Draw Fig. S2 a-c for 16 fields

In [ ]:
x = range(1950 , 2017)
c = 'TeamAgeMax'
y = dfp.groupby('Year')[c].mean()
ylst = [dfp[dfp['FieldName']==F].groupby('Year')[c].mean() for F in Flst]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
sns.lineplot(dfp,x='Year',y=c,color='grey',n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.plot(y.index, y.values,linewidth=3,alpha = 1,c='black')
for i in range(len(ylst)):
    plt.plot(ylst[i].index, ylst[i].values,linewidth=1,alpha = .4,c=colors[i])
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Max Age',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(1,30)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/a_2.svg'),bbox_inches = 'tight')

In [ ]:
x = range(1950 , 2017)
c2 = 'TeamAgeMin'
y2 = dfp.groupby('Year')[c2].mean()
ylst2 = [dfp[dfp['FieldName']==F].groupby('Year')[c2].mean() for F in Flst]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
sns.lineplot(dfp,x='Year',y=c2,color='grey',n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.plot(y2.index, y2.values,linewidth=3,alpha = 1,c='black')
for i in range(len(ylst2)):
    plt.plot(ylst2[i].index, ylst2[i].values,linewidth=1,alpha = .4,c=colors[i])
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Min Age',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(1,10)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/b_2.svg'),bbox_inches = 'tight')

In [ ]:
d1 = pd.DataFrame(dfp.groupby(dfp['FieldName'])[c].mean())  
d2 = pd.DataFrame(dfp.groupby(dfp['FieldName'])[c2].mean())
d1 = d1.loc[Flst,:]
d2 = d2.loc[Flst,:]

In [ ]:
plt.subplots(1,2,figsize=(5,7))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.2,hspace=0)
ax = plt.subplot(1,2, 1)
plt.barh( d1.index ,  width=d1.values.flatten(), capsize = 3, color = colors)
plt.ylim(-.7,15.7)
plt.yticks(fontsize=20,)
plt.xticks([5,10,15,20],fontsize=20,)
plt.title('Team Max Age  ',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = plt.subplot(1,2, 2)
plt.barh( d2.index ,  width=d2.values.flatten(), capsize = 3, color = colors)
plt.ylim(-.7,15.7)
plt.yticks(fontsize=0,)
plt.xticks([2,4,6,8],fontsize=20,)
plt.title('  Team Min Age',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/c_2.svg'),bbox_inches = 'tight')

# # Draw Fig. 1 d-e for different team sizes

In [ ]:
x = range(1950 , 2017)
c = 'TeamAgeMax'
ylst3 = [dfp[dfp['TeamSizeG']==G].groupby('Year')[c].mean() for G in range(1,10)]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
for i in range(len(ylst3)):
    plt.plot(ylst3[i].index, ylst3[i].values,linewidth=2,alpha = 1,c=colors2[i])
    sns.lineplot(dfp[dfp['TeamSizeG']==i+1],x='Year',y=c,color=colors2[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Max Age',fontsize = 20)
plt.xlim(1950,2016)
plt.ylim(1,30)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/d.svg'),bbox_inches = 'tight')

In [ ]:
x = range(1950 , 2017)
c2 = 'TeamAgeMin'
ylst4 = [dfp[dfp['TeamSizeG']==G].groupby('Year')[c2].mean() for G in range(1,10)]

In [ ]:
f, ax = plt.subplots(figsize=(6, 5))
for i in range(len(ylst4)):
    plt.plot(ylst4[i].index, ylst4[i].values,linewidth=2,alpha = 1,c=colors2[i])
    sns.lineplot(dfp[dfp['TeamSizeG']==i+1],x='Year',y=c2,color=colors2[i],n_boot=BOOTSTRAP_RESAMPLES, seed=RANDOM_SEED)
plt.yticks(fontsize=20,)
plt.xticks([1950,1970,1990,2010],fontsize=20,)
plt.xlabel('Year',fontsize = 20)
plt.ylabel('Team Min Age',fontsize = 20)
plt.ylim(1,10)
plt.xlim(1950,2016)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
ax.spines['right'].set_linewidth(2)
ax.spines['top'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/e_2.svg'),bbox_inches = 'tight')

In [ ]:
d11 = pd.DataFrame(dfp.groupby(dfp['TeamSizeG'])[c].mean())  
d21 = pd.DataFrame(dfp.groupby(dfp['TeamSizeG'])[c2].mean())

In [ ]:
plt.subplots(1,2,figsize=(5,7))
plt.subplots_adjust(left=None,bottom=None,right=None,top=None,wspace=0.2,hspace=0)
ax = plt.subplot(1,2, 1)
plt.barh( d11.index ,  width=d11.values.flatten(), capsize = 3, color = colors2)
plt.xlim(6,14)
plt.ylim(-.7,8.7)
plt.yticks(fontsize=20,)
plt.xticks([10,20,30,40],fontsize=20,)
plt.title('Team Max Age  ',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax = plt.subplot(1,2, 2)
plt.barh( d21.index ,  width=d21.values.flatten(), capsize = 3, color = colors2)
plt.xlim(0.6,1)
plt.ylim(-.7,8.7)
plt.yticks(fontsize=0,)
plt.xticks([3,6,9,12],fontsize=20,)
plt.title('  Team Min Age',fontsize = 20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)
plt.savefig(output_path('fig_1_S2/f_2.svg'),bbox_inches = 'tight')